## Video Generation Notebook
### For Tracker Data
#### Helper Functions for creating mp4s and GIFs of tracker images

In [7]:
import os
import cv2
import glob
from pathlib import Path
from PIL import Image
import numpy as np
from tkinter import filedialog
import tkinter as tk
from pathlib import Path


In [ ]:
def create_gif_pillow(input_folder, output_path, duration=100):
    """
    Create GIF animation using PIL/Pillow
    
    Args:
        input_folder: Path to folder containing images
        output_path: Output GIF file path
        duration: Duration between frames in milliseconds
    """
    # Get all jpg files and sort them numerically
    pattern = os.path.join(input_folder, "*.jpg")
    image_files = glob.glob(pattern)
    
    # Sort by the first number in filename (1_0, 2_1, 3_2, etc.)
    image_files.sort(key=lambda x: int(os.path.basename(x).split('_')[0]))
    
    if not image_files:
        print("No JPG files found in the specified folder!")
        return
    
    # Load images
    images = []
    for file in image_files:
        img = Image.open(file)
        images.append(img)
    
    # Save as GIF
    images[0].save(
        output_path,
        save_all=True,
        append_images=images[1:],
        duration=duration,
        loop=0  # 0 means infinite loop
    )
    print(f"GIF saved to: {output_path}")

def create_mp4_opencv(input_folder, output_path, fps=30):
    """
    Create MP4 video using OpenCV
    
    Args:
        input_folder: Path to folder containing images
        output_path: Output MP4 file path
        fps: Frames per second
    """
    pattern = os.path.join(input_folder, "*.jpg")
    image_files = glob.glob(pattern)
    image_files.sort(key=lambda x: int(os.path.basename(x).split('_')[0]))
    
    if not image_files:
        print("No JPG files found!")
        return
    
    # Read first image to get dimensions
    first_image = cv2.imread(image_files[0])
    height, width, layers = first_image.shape
    
    # Define codec and create VideoWriter
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Add each image to video
    for file in image_files:
        img = cv2.imread(file)
        video.write(img)
    
    # Release everything
    video.release()
    cv2.destroyAllWindows()
    print(f"MP4 saved to: {output_path}")

#### Example Use Case for One Folder

In [ ]:
# Your specific folder path
submodule = os.path.expanduser("~") + "/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/"
folder_name= "08_19_2025_11_31_41_Medium_Turtle_Track"
input_folder = submodule + folder_name + "/track/Tracker-result"
output_folder = submodule + folder_name + "/track/animations/"
os.makedirs(output_folder, exist_ok=True)
# Check if folder exists
if not os.path.exists(input_folder):
    print(f"Folder not found: {input_folder}")
    exit()

# Create different animation formats
print("Creating animations...")

# GIF using PIL (good for web sharing)
# create_gif_pillow(input_folder, "animation_pillow.gif", duration=100)

# MP4 using OpenCV (high quality video)
create_mp4_opencv(input_folder, output_folder + folder_name + ".mp4", fps=15)

print("All animations created successfully!")

#### Tracker Video Generation

In [ ]:
# Your specific folder path
submodule = os.path.expanduser("~") + "/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/"

    # Find all folders starting with "08_19_2025"
folders_to_process = []
if os.path.exists(submodule):
    for folder in os.listdir(submodule):
        if folder.startswith("08_19_2025") and os.path.isdir(os.path.join(submodule, folder)):
            folders_to_process.append(folder)

for folder_name in folders_to_process:
    print(f"Processing folder: {folder_name}")
    input_folder = submodule + folder_name + "/track/Tracker-result"
    output_folder = submodule + folder_name + "/track/animations/"
    os.makedirs(output_folder, exist_ok=True)
    # Check if folder exists
    if not os.path.exists(input_folder):
        print(f"Folder not found: {input_folder}")
        exit()

    # Create different animation formats
    print("Creating mp4...")
    # MP4 using OpenCV (high quality video)
    create_mp4_opencv(input_folder, output_folder + folder_name + ".mp4", fps=15)
    print("Creating gif...")
    # GIF using PIL (good for web sharing)
    create_gif_pillow(input_folder, "animation_pillow.gif", duration=100)

    print(f"All animations created successfully for {folder_name}!")


Processing folder: 08_19_2025_10_41_08
Creating mp4...
MP4 saved to: /home/flyingwhale/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/08_19_2025_10_41_08/track/animations/08_19_2025_10_41_08.mp4
Creating gif...
GIF saved to: animation_pillow.gif
All animations created successfully for 08_19_2025_10_41_08!
Processing folder: 08_19_2025_10_14_57_Stingray_Turtle
Creating mp4...
MP4 saved to: /home/flyingwhale/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/08_19_2025_10_14_57_Stingray_Turtle/track/animations/08_19_2025_10_14_57_Stingray_Turtle.mp4
Creating gif...
GIF saved to: animation_pillow.gif
All animations created successfully for 08_19_2025_10_14_57_Stingray_Turtle!
Processing folder: 08_19_2025_11_05_23
Creating mp4...
MP4 saved to: /home/flyingwhale/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/08_19_2025_11_05_23/track/animations/08_19_2025_11_05_23.mp4
Creating gif...
GIF saved to: animation_pillow.gif
All animations created successfull

: 

### Video Generation for Left/Right Images
#### Helper Functions

In [ ]:
def natural_sort_key(s):
    """
    Sort strings containing numbers naturally (e.g., frame1.npy, frame2.npy, ..., frame10.npy)
    """
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

def create_video_from_images(image_folder, output_path, fps=15, extension="jpg"):
    """
    Create a video from a folder of images
    
    Args:
        image_folder (str): Path to folder containing images
        output_path (str): Path where to save the output video
        fps (int): Frames per second for the output video
        extension (str): File extension of the images to use
    
    Returns:
        bool: True if successful, False otherwise
    """
    # Check if folder exists
    if not os.path.exists(image_folder):
        print(f"Error: Directory '{image_folder}' not found.")
        return False
    
    # Find all image files in the directory
    image_files = glob.glob(os.path.join(image_folder, f"*.{extension}"))
    
    if not image_files:
        print(f"Error: No {extension} files found in '{image_folder}'")
        return False
    
    # Sort files naturally
    image_files.sort(key=natural_sort_key)
    print(f"Found {len(image_files)} images in '{image_folder}'")
    
    # Read first image to get dimensions
    first_image = cv2.imread(image_files[0])
    if first_image is None:
        print(f"Error: Could not read image '{image_files[0]}'")
        return False
    
    height, width, channels = first_image.shape
    
    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Use 'XVID' for .avi format
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Process each image
    print(f"Creating video: {output_path}")
    for image_file in tqdm(image_files):
        image = cv2.imread(image_file)
        if image is not None:
            video_writer.write(image)
    
    # Release the video writer
    video_writer.release()
    print(f"Video saved to {output_path}")
    
    return True

def create_stereo_videos(left_folder, right_folder, output_dir=None, fps=15, extension="jpg"):
    """
    Create separate videos from left and right image folders
    
    Args:
        left_folder (str): Path to folder containing left camera images
        right_folder (str): Path to folder containing right camera images
        output_dir (str): Directory to save output videos (default: current directory)
        fps (int): Frames per second for the output videos
        extension (str): File extension of the images to use
    """
    # Use current directory if output_dir is not specified
    if output_dir is None:
        output_dir = os.getcwd()
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Define output paths
    left_video_path = os.path.join(output_dir, "left_camera.mp4")
    right_video_path = os.path.join(output_dir, "right_camera.mp4")
    
    # Create videos
    left_success = create_video_from_images(left_folder, left_video_path, fps, extension)
    right_success = create_video_from_images(right_folder, right_video_path, fps, extension)
    
    # Check results
    if left_success and right_success:
        print("\nBoth videos created successfully:")
        print(f"Left camera video: {left_video_path}")
        print(f"Right camera video: {right_video_path}")
    else:
        print("\nThere were issues creating one or both videos.")

def interactive_stitch(self, left, right):
    h, w = left.shape[:2]

    corners = np.array([
        [0, 0],
        [0, h],
        [w, 0],
        [w, h]
    ], dtype=np.float32)

    transformed_corners = cv2.transform(np.array([corners]), self.affine_matrix)[0]
    all_corners = np.vstack((corners, transformed_corners))

    [xmin, ymin] = np.floor(all_corners.min(axis=0)).astype(int)
    [xmax, ymax] = np.ceil(all_corners.max(axis=0)).astype(int)
    output_size = (xmax - xmin, ymax - ymin)
    offset = np.array([-xmin, -ymin])

    affine_with_offset = self.affine_matrix.copy()
    affine_with_offset[:, 2] += offset

    transformed_right = cv2.warpAffine(right, affine_with_offset, output_size)

    canvas = np.zeros((output_size[1], output_size[0], 3), dtype=np.uint8)
    canvas[offset[1]:offset[1] + h, offset[0]:offset[0] + w] = left

    stitched = np.maximum(canvas, transformed_right)

    return stitched

def stitch_left_right_images(left_image_path, right_image_path, output_path):
    """
    Stitch left and right images side by side
    
    Args:
        left_image_path (str): Path to the left image
        right_image_path (str): Path to the right image
        output_path (str): Path to save the stitched image
    
    Returns:
        bool: True if successful, False otherwise
    """
    # Read images
    left_image = cv2.imread(left_image_path)
    right_image = cv2.imread(right_image_path)
    
    if left_image is None or right_image is None:
        print("Error: Could not read one or both images.")
        return False
    
    # Resize images to the same height
    height = min(left_image.shape[0], right_image.shape[0])
    left_image_resized = cv2.resize(left_image, (int(left_image.shape[1] * height / left_image.shape[0]), height))
    right_image_resized = cv2.resize(right_image, (int(right_image.shape[1] * height / right_image.shape[0]), height))
    
    # Concatenate images side by side
    stitched_image = np.hstack((left_image_resized, right_image_resized))
    
    # Save the stitched image
    cv2.imwrite(output_path, stitched_image)
    print(f"Stitched image saved to {output_path}")

#### Example Use Case for One Folder

In [ ]:
root = tk.Tk()
root.withdraw()  # Hide the main window

folder_path = filedialog.askdirectory(
    title="Select your video folder",
    initialdir=os.path.expanduser("~") + "/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/"
)

if folder_path:  # Check if a file was selected
    # Convert to Path object for easier manipulation
    folder_path_obj = Path(folder_path)
    left_dir = folder_path_obj / "left"
    right_dir = folder_path_obj / "right"    
    print("Selected folder:", folder_path)
    create_video_from_images
else:
    print("No folder selected")
    folder_path = ""
    npz_filename = ""

Selected folder: /home/ranger/drl-turtle/ros2_ws/src/turtle_hardware/turtle_hardware/video/06_16_2025_15_06_19
No .npz files found in the selected folder
